<a href="https://colab.research.google.com/github/davidfashokun/SRS_Req_Classification_Promise_Phi3.5/blob/main/srs_req_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl
import pandas as pd
from datasets import Dataset
import io

# Manually parse the ARFF file content to handle string attributes
arff_file_path = "/content/dataset/PROMISE_exp.arff"
data_lines = []
attributes = []
in_data_section = False

with open(arff_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line.lower().startswith('@data'):
            in_data_section = True
            continue
        if line.lower().startswith('@attribute') and not in_data_section:
            # Parse attribute line to get column names
            parts = line.split()
            attr_name = parts[1]
            attributes.append(attr_name)
        elif in_data_section and line:
            data_lines.append(line)

# Create a string buffer for pandas to read the data section
data_string = io.StringIO("\n".join(data_lines))

# Read the data into a DataFrame, assuming comma separation. If this fails, we will need to inspect the delimiter.
df = pd.read_csv(data_string, header=None, names=attributes)

# Format into instruction-response pairs
def format_example(row):
    return {
        "text": f"""<|user|>
Classify the following software requirement into one of these categories:
Functional (F), Usability (US), Security (SE), Operational (O), Performance (PE),
Look & Feel (LF), Maintainability (MN), Portability (PO), Legal (L), Fault Tolerance (FT), Scalability (SC), Availability (A).

Requirement: {row['RequirementText']}<|end|>
<|assistant|>
{row['_class_']}<|end|>"""
    }

dataset = Dataset.from_pandas(df).map(format_example)
dataset = dataset.train_test_split(test_size=0.2)

Map:   0%|          | 0/969 [00:00<?, ? examples/s]

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [3]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You'll see something like: ~0.5% of params are trainable

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./phi3.5-promise",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    eval_strategy="epoch", # new args below
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Attach tokenizer to the model directly if SFTTrainer doesn't accept it as an argument
model.tokenizer = tokenizer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    # tokenizer=tokenizer,  # Removed as it's causing the TypeError
    # dataset_text_field="text",  # Removed as it's causing a TypeError
    # max_length=512, # Removed as it's causing the TypeError
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/775 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/775 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/194 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/194 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is de

Step,Training Loss
10,10.203790
20,8.002379
30,6.005458
40,3.798742
50,2.413700
60,1.978490
70,1.776769
80,1.624740
90,1.592927
100,1.568799


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=291, training_loss=2.2677412119108378, metrics={'train_runtime': 3629.8738, 'train_samples_per_second': 0.641, 'train_steps_per_second': 0.08, 'total_flos': 6516650120896512.0, 'train_loss': 2.2677412119108378})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training Loss,Validation Loss,Step
1.313462,11.131411,291


{'eval_loss': 11.131410598754883}

In [18]:
from sklearn.metrics import classification_report
import torch

model.eval()
predictions, true_labels = [], []

for example in dataset["test"]:
    # Build prompt without the answer
    prompt = example["text"].split("<|assistant|>")[0] + "<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False, use_cache=False)

    pred = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    true = example["text"].split("<|assistant|>")[1].replace("<|end|>", "").strip()

    predictions.append(pred)
    true_labels.append(true)

print(classification_report(true_labels, predictions))

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

                                                                precision    recall  f1-score   support

                                                             A       0.00      0.00      0.00       6.0
                       AddAdd Ext Ext Extmer Ext Ext Moore Ext       0.00      0.00      0.00       0.0
                Adrilookup Ext Ext Moore Ext Ext Ext Leo siehe       0.00      0.00      0.00       0.0
               Dictionary Ext siehe Extwohl ужеwohl Extmer Ext       0.00      0.00      0.00       0.0
                       Ext Ext Ext Ext Ext Ext Ext Ext Ext Ext       0.00      0.00      0.00       0.0
                   Ext Ext Ext Ext Ext Ext Ext Ext Ext genannt       0.00      0.00      0.00       0.0
                     Ext Ext Ext Ext Ext Ext Ext Ext Extlookup       0.00      0.00      0.00       0.0
                     Ext Ext Ext Ext Ext Moore Ext Ext Ext Ext       0.00      0.00      0.00       0.0
                     Ext Ext Ext Ext Moore Ext Ext Ext Ext Ext 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [19]:
!zip -r phi3.5-promise-final.zip ./phi3.5-promise-final
from google.colab import files
files.download("phi3.5-promise-final.zip")

  adding: phi3.5-promise-final/ (stored 0%)
  adding: phi3.5-promise-final/chat_template.jinja (deflated 61%)
  adding: phi3.5-promise-final/tokenizer.json (deflated 85%)
  adding: phi3.5-promise-final/adapter_config.json (deflated 58%)
  adding: phi3.5-promise-final/training_args.bin (deflated 53%)
  adding: phi3.5-promise-final/tokenizer_config.json (deflated 46%)
  adding: phi3.5-promise-final/README.md (deflated 65%)
  adding: phi3.5-promise-final/adapter_model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# Save the LoRA adapter
trainer.save_model("./phi3.5-promise-final")

# Merge LoRA into base model so you can use it standalone
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig # Import necessary classes
import torch

# Re-define bnb_config for robustness if it's not globally available within this cell's scope
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Re-load the original base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, # model_id is 'microsoft/Phi-3.5-mini-instruct' from previous cell
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load the PeftModel (adapter) onto the base model
merged_model = PeftModel.from_pretrained(base_model, "./phi3.5-promise-final")

# Merge the LoRA weights into the base model and unload the adapters
merged_model = merged_model.merge_and_unload()

# Save the merged model and tokenizer
merged_model.save_pretrained("./phi3.5-promise-merged", save_original_format=True)
tokenizer.save_pretrained("./phi3.5-promise-merged")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.3.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.3.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.4.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.4.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.5.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.5.self_attn.o_proj.l

AttributeError: 'list' object has no attribute 'keys'